# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cjalanano-dev/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The model outputs a probability of decline. We combine this score with simple heuristics to assign a **Reason Code** and an **Action**:
- **Reason Code: Stale & High Value** (if `days_since_last_update` > 90 and high impressions) -> **Action: Deep Refresh**
- **Reason Code: Slipping CTR** (if `ctr` is lower than median for its position) -> **Action: Title & Meta Optimization**
- **Reason Code: Thin Content** (if `word_count` < 500) -> **Action: Content Expansion**
- Otherwise -> **Action: Standard Review**

In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer

# Load Data
import pathlib
try:
    notebook_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    notebook_dir = os.getcwd()

if notebook_dir.endswith('notebooks'):
    REPO_ROOT = os.path.abspath(os.path.join(notebook_dir, '..', '..'))
else:
    REPO_ROOT = os.path.abspath(notebook_dir)

DATA_PATH = os.path.join(REPO_ROOT, 'data', 'raw', 'content_refresh_anonymized.csv')
df = pd.read_csv(DATA_PATH)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
df['avg_position'] = df['avg_position'].replace(0, np.nan)

# Same features as our audited model
raw_features = ['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 
                'days_since_last_update', 'word_count', 'engagement_rate', 
                'scroll_rate', 'ai_traffic_pct']
for col in ['word_count', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']:
    df[f'has_{col}'] = df[col].notna().astype(int)

features = raw_features + [f'has_{col}' for col in ['word_count', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']]
imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(df[features]), columns=features)
y = df['is_declining_label']

# Train full model to score the queue
rf = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, n_jobs=-1)
rf.fit(X, y)
df['ml_decline_score'] = rf.predict_proba(X)[:, 1]

# Apply Playbook Logic
def assign_action(row):
    if row['days_since_last_update'] > 90 and row['impressions_90d'] > 1000:
        return 'Deep Refresh', 'Stale & High Value'
    elif row['word_count'] < 500:
        return 'Content Expansion', 'Thin Content'
    elif row['ctr'] < 2.0 and row['avg_position'] <= 10:
        return 'Title & Meta Optimization', 'Slipping CTR on Page 1'
    return 'Standard Review', 'Model Flagged'

# Score and rank
queue = df.sort_values('ml_decline_score', ascending=False).copy()
queue['action'], queue['reason_code'] = zip(*queue.apply(assign_action, axis=1))

print("Top 5 Actionable Content Pieces:")
print(queue[['content_id', 'ml_decline_score', 'action', 'reason_code']].head(5).to_string(index=False))


Top 5 Actionable Content Pieces:
          content_id  ml_decline_score          action   reason_code
content_acf71f98ada2          0.764887 Standard Review Model Flagged
content_41b9311ed497          0.764465 Standard Review Model Flagged
content_cb3c920a1bfd          0.763338 Standard Review Model Flagged
content_fc7c3c7271cb          0.761481 Standard Review Model Flagged
content_9d9905bcb297          0.760628 Standard Review Model Flagged


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended Use:** This queue is designed as a **decision-support tool** for content strategists and SEO managers. It ranks thousands of pages to highlight those exhibiting signals directionally associated with traffic decay, so humans know where to look first.

**Limits:**
- It **does not prove** that fixing a page will recover traffic.
- The model relies heavily on historical search console signals (like impressions and CTR); it cannot measure external factors like Google core updates or shifts in competitor strategy.
- It is invalid for newly published content (<90 days old) as they lack sufficient historical data.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Human Review Checklist:**
- Verify if the page's topic is still relevant to the business (if not, consider deprecation instead of refresh).
- Read the content to see *why* the CTR might be slipping (e.g., outdated title year).

**The No-Go List (Never Automate):**
- **Never automate direct content generation/publishing** based solely on these scores. A high score means "review this," not "rewrite this with AI and push to production." 
- Do not blindly change URLs or delete content without mapping redirects.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Triggers to Retrain or Investigate:**
- The overall Precision@100 on new validation windows drops below our initial benchmark (0.53).
- A major Google algorithm update shifts the global base rate of declining content by more than 10%.
- Over 50% of the top-ranked recommendations are rejected by human reviewers for being "False Alarms."


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:
import json

# Ensure directories exist
OUTPUT_DIR = os.path.join(REPO_ROOT, 'work', 'outputs')
FIGURE_DIR = os.path.join(REPO_ROOT, 'work', 'figures')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FIGURE_DIR, exist_ok=True)

# 1. Export Queue
csv_path = os.path.join(OUTPUT_DIR, 'action_queue.csv')
queue.to_csv(csv_path, index=False)

# 2. Export Figure
plt.figure(figsize=(8, 5))
plt.hist(queue['ml_decline_score'], bins=50, color='skyblue', edgecolor='black')
plt.title('Distribution of ML Decline Scores')
plt.xlabel('Probability of Decline')
plt.ylabel('Number of Content Pieces')
plt.grid(axis='y', alpha=0.75)
fig_path = os.path.join(FIGURE_DIR, 'score_distribution.png')
plt.savefig(fig_path, bbox_inches='tight')
plt.close()

# 3. Export Metrics Receipt
metrics = {
    'total_scored': len(queue),
    'top_100_avg_score': float(queue['ml_decline_score'].head(100).mean()),
    'actions_breakdown': queue['action'].value_counts().to_dict()
}
json_path = os.path.join(OUTPUT_DIR, 'w07_playbook_metrics.json')
with open(json_path, 'w') as f:
    json.dump(metrics, f, indent=2)

print(f"Exported queue to {csv_path}")
print(f"Exported figure to {fig_path}")
print(f"Exported metrics to {json_path}")


Exported queue to C:\Users\carlosjames\Projects\flyrank-ml-internship\work\outputs\action_queue.csv
Exported figure to C:\Users\carlosjames\Projects\flyrank-ml-internship\work\figures\score_distribution.png
Exported metrics to C:\Users\carlosjames\Projects\flyrank-ml-internship\work\outputs\w07_playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.